In [1]:
import numpy as np
import pandas as pd
import keras
from keras import ops
from keras import layers
import matplotlib.pyplot as plt
import keras_hub
from sklearn.model_selection import train_test_split

In [2]:
def load_data_and_encode(dataset_path, nuc_to_idx, max_rows):
    """
    Read the Ribonanza CSV file, keep the sequences measured with both
    experiments (DMS_MaP and 2A3_MaP) and encode each nucleotide sequence
    as a list of integer indices.

    Parameters
    ----------
    dataset_path : str
        Path to the CSV file (train_data_QUICK_START.csv). Each sequence
        appears on two rows, one per experiment_type.
    nuc_to_idx : dict
        Mapping from nucleotide character (A/C/G/U) to integer index.
    max_rows : int or None
        Number of sequences to keep (useful to test quickly).
        None keeps all the sequences.

    Returns
    -------
    df_dms : pd.DataFrame
        Rows of the DMS_MaP experiment, sorted by sequence_id.
    df_2a3 : pd.DataFrame
        Rows of the 2A3_MaP experiment, sorted by sequence_id.
        Row i of df_dms and row i of df_2a3 are the same sequence.
    sequences : list of str
        Nucleotide sequences as strings.
    encoded_seqs : list of list of int
        Nucleotide sequences encoded as lists of integer indices.
    """
    sequence_dataset = pd.read_csv(dataset_path)

    # We split the two experiments in two tables
    df_dms = sequence_dataset[sequence_dataset["experiment_type"] == "DMS_MaP"]
    df_2a3 = sequence_dataset[sequence_dataset["experiment_type"] == "2A3_MaP"]

    # We keep only the sequences which have the two measures
    ids_dms = set(df_dms["sequence_id"])
    ids_2a3 = set(df_2a3["sequence_id"])
    common_ids = sorted(ids_dms & ids_2a3)

    if max_rows is not None:
        common_ids = common_ids[:max_rows]

    # We sort by sequence_id so that the two tables are in the same order
    df_dms = df_dms[df_dms["sequence_id"].isin(common_ids)]
    df_dms = df_dms.drop_duplicates(subset="sequence_id")
    df_dms = df_dms.sort_values("sequence_id")

    df_2a3 = df_2a3[df_2a3["sequence_id"].isin(common_ids)]
    df_2a3 = df_2a3.drop_duplicates(subset="sequence_id")
    df_2a3 = df_2a3.sort_values("sequence_id")

    # The sequences are the same in the two tables
    sequences = df_dms["sequence"].astype(str).tolist()

    # We want to turn letters into numbers before putting them into a vector
    encoded_seqs = []

    # We map each letter to its number
    for seq in sequences:
        encoded_seq = []

        for c in seq:
            encoded_seq.append(nuc_to_idx.get(c, 0))

        encoded_seqs.append(encoded_seq)

    return df_dms, df_2a3, sequences, encoded_seqs

In [3]:
def build_targets(df_dms, df_2a3, maxlen):
    """
    Build the target array with the two reactivities of each position.

    Parameters
    ----------
    df_dms : pd.DataFrame
        Rows of the DMS_MaP experiment (output of load_data_and_encode).
    df_2a3 : pd.DataFrame
        Rows of the 2A3_MaP experiment, in the same order as df_dms.
    maxlen : int
        Number of positions kept for each sequence.

    Returns
    -------
    y : np.ndarray of shape (n_samples, maxlen, 2)
        Reactivities clipped between 0 and 1. Index 0 of the last axis is
        DMS_MaP and index 1 is 2A3_MaP (same order as the submission file).
        NaN means "no measure at this position".
    """
    reactivity_cols = []
    for i in range(1, maxlen + 1):
        reactivity_cols.append("reactivity_" + str(i).zfill(4))

    y_dms = df_dms[reactivity_cols].to_numpy()
    y_2a3 = df_2a3[reactivity_cols].to_numpy()

    # Values must be between 0 and 1 (sigmoid output, and same as the Kaggle
    # score). NaN stay NaN.
    y_dms = np.clip(y_dms, 0, 1)
    y_2a3 = np.clip(y_2a3, 0, 1)

    # We put the two measures side by side on the last axis
    y = np.stack([y_dms, y_2a3], axis=-1)

    return y

In [4]:
def prepare_data(encoded_seqs, y, maxlen):
    """
    Pad the encoded sequences to a fixed length, then split the dataset
    into training and test sets.

    Parameters
    ----------
    encoded_seqs : list of list of int
        Nucleotide sequences encoded as lists of integer indices.
    y : np.ndarray of shape (n_samples, maxlen, 2)
        Targets (output of build_targets).
    maxlen : int
        Fixed length to which every sequence is padded or truncated.

    Returns
    -------
    X : np.ndarray of shape (n_samples, maxlen)
        Padded encoded sequences.
    x_train, x_test : np.ndarray
        Train/test split of X.
    y_train, y_test : np.ndarray
        Train/test split of y.
    """
    X = keras.utils.pad_sequences(
        encoded_seqs,
        maxlen=maxlen,
        padding="post",
        truncating="post",
        value=0,
    )

    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    return X, x_train, x_test, y_train, y_test

In [5]:
def create_model(maxlen, embed_dim, vocab_size, num_outputs, num_heads, dense_neuron, num_block_treatment):
    """
    Build the model architecture.

    Parameters
    ----------
    maxlen : int
        Fixed length of the input sequences (after padding/truncation).
    embed_dim : int
        Size of the embeddings (token + position) and attention key size.
    vocab_size : int
        Size of the nucleotide vocabulary (number of distinct indices).
    num_outputs : int
        Number of values predicted for each position (2: DMS_MaP, 2A3_MaP).
    num_heads : int
        Number of attention heads in each MultiHeadAttention block.
    dense_neuron : int
        Number of neurons of the intermediate dense layer (before the output).
    num_block_treatment : int
        Number of residual blocks (normalization + self-attention) stacked.

    Returns
    -------
    model : keras.Model
        Model not compiled, taking a batch of encoded sequences of shape
        (batch_size, maxlen) and giving reactivities between 0 and 1 of
        shape (batch_size, maxlen, num_outputs).
    """
    inputs = keras.Input(shape=(maxlen,))
    x_emb = keras_hub.layers.TokenAndPositionEmbedding(
        vocabulary_size=vocab_size,
        sequence_length=maxlen,
        embedding_dim=embed_dim,
    )(inputs)
    x = x_emb
    for i in range(num_block_treatment):
        # Partie 1 : self-attention
        x_norm = layers.LayerNormalization(epsilon=1e-6)(x)
        x_atn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )(x_norm, x_norm)
        x_atn = layers.Dropout(0.1)(x_atn)             # ← nouveau
        x = x + x_atn

        # Partie 2 : feed-forward
        x_norm2 = layers.LayerNormalization(epsilon=1e-6)(x)
        x_ff = layers.Dense(embed_dim * 4, activation="gelu")(x_norm2)  # ← GELU
        x_ff = layers.Dropout(0.1)(x_ff)                # ← nouveau
        x_ff = layers.Dense(embed_dim)(x_ff)
        x_ff = layers.Dropout(0.1)(x_ff)                # ← nouveau
        x = x + x_ff
    dens = layers.Dense(dense_neuron)(x)
    outputs = layers.Dense(num_outputs, activation="sigmoid")(dens)
    model = keras.Model(inputs=inputs, outputs=outputs)

    return model


In [6]:
def masked_mae(y_true, y_pred):
    """
    Mean absolute error like the Kaggle score: one single mean over all the
    measured values. The NaN of y_true (padding, no measure) are ignored.

    Parameters
    ----------
    y_true : tensor of shape (batch_size, maxlen, 2)
        True reactivities, NaN where there is no value.
    y_pred : tensor of shape (batch_size, maxlen, 2)
        Reactivities predicted by the model.

    Returns
    -------
    mae : tensor
        One number: the sum of all the errors divided by the number of
        measured values in the batch.
    """
    # mask = 1 if there is a real value, 0 if NaN
    mask = ops.cast(ops.logical_not(ops.isnan(y_true)), "float32")

    # We replace the NaN by the prediction, so the error is 0 at this place
    y_true = ops.where(mask > 0, y_true, y_pred)
    erreur = ops.abs(y_true - y_pred) * mask

    return ops.sum(erreur) / (ops.sum(mask) + 1e-7)

In [7]:
def train_model(model, x_train, y_train, batch_size, epoch_num, patience):
    """
    Compile and train the model.

    Parameters
    ----------
    model : keras.Model
        Model not compiled to train (output of create_model).
    x_train : np.ndarray
        Encoded and padded sequences, shape (n_samples, maxlen).
    y_train : np.ndarray
        Reactivities with NaN where there is no value,
        shape (n_samples, maxlen, 2).
    batch_size : int
        Size of the batches used during training.
    epoch_num : int
        Maximum number of epochs.
    patience : int
        Number of epochs without improvement of val_loss accepted before
        the early stopping.

    Returns
    -------
    history : keras.callbacks.History
        Training history with the loss (train and validation) of each epoch.
    """
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=3e-4),
        loss=masked_mae,
        metrics=[mae_dms, mae_2a3],
    )

    callbacks = [
        keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_delta=1e-4,
        cooldown=2,
        min_lr=1e-5,
        verbose=1,
    ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
        )
    ]

    history = model.fit(
        x_train,
        y_train,
        batch_size=batch_size,
        epochs=epoch_num,
        validation_split=0.2,
        callbacks=callbacks,
    )
    return history


In [8]:
def plot_history(history, model_name):

    out_path = f"training_loss_{model_name}.png"

    train_loss = history.history.get("loss", [])
    val_loss = history.history.get("val_loss", [])

    epochs = range(1, len(train_loss) + 1)

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_loss, label="Train loss")
    plt.plot(epochs, val_loss, label="Val loss", linestyle="--")
    plt.title("Loss (MAE)")
    plt.xlabel("Epochs")
    plt.ylabel("MAE")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Courbe sauvegardée sous : {out_path}")

In [9]:
def evaluate_model(model, x_test, y_test, model_name):
    """
    Evaluate the trained model on the held-out test set and print the MAE
    computed like on Kaggle (one single mean over all the measured values).

    Parameters
    ----------
    model : keras.Model
        The trained model to evaluate.
    x_test : np.ndarray
        Encoded and padded test sequences.
    y_test : np.ndarray
        Test reactivities, shape (n_samples, maxlen, 2), NaN where no value.
    model_name : str
        Name used to identify this model configuration in the printed output.

    Returns
    -------
    results : dict
        Dictionary with the keys "mae" (both measures), "mae_dms" and
        "mae_2a3".
    """
    pred = model.predict(x_test)

    # Both measures together (this is the Kaggle score)
    mask_all = ~np.isnan(y_test)
    mae_all = np.abs(y_test[mask_all] - pred[mask_all]).mean()

    # DMS_MaP alone
    mask_dms = ~np.isnan(y_test[:, :, 0])
    mae_dms = np.abs(y_test[:, :, 0][mask_dms] - pred[:, :, 0][mask_dms]).mean()

    # 2A3_MaP alone
    mask_2a3 = ~np.isnan(y_test[:, :, 1])
    mae_2a3 = np.abs(y_test[:, :, 1][mask_2a3] - pred[:, :, 1][mask_2a3]).mean()

    results = {"mae": mae_all, "mae_dms": mae_dms, "mae_2a3": mae_2a3}
    print(f"[{model_name}] Test MAE: {mae_all:.4f} — DMS_MaP: {mae_dms:.4f} — 2A3_MaP: {mae_2a3:.4f}")
    return results

In [10]:
def mae_dms(y_true, y_pred):
    return masked_mae(y_true[:, :, 0:1], y_pred[:, :, 0:1])

def mae_2a3(y_true, y_pred):
    return masked_mae(y_true[:, :, 1:2], y_pred[:, :, 1:2])

In [11]:
def predict_random_sequence(model, sequences, maxlen, nuc_to_idx):
    seq = sequences[np.random.randint(len(sequences))]

    print("\n=== Exemple tiré au hasard ===")
    print("Séquence :", seq)

    encoded = [nuc_to_idx.get(c, 0) for c in seq]
    encoded = keras.utils.pad_sequences(
        [encoded], maxlen=maxlen, padding="post", truncating="post"
    )

    pred = model.predict(encoded)[0]

    # We show only the real positions (not the padding)
    print("Réactivités DMS_MaP prédites :", pred[:len(seq), 0])
    print("Réactivités 2A3_MaP prédites :", pred[:len(seq), 1])
    print("=============================\n")

    return seq, pred[:len(seq)]

In [12]:
def plot_sequence_length_distribution(df, maxlen_ref):
    """
    Show the distribution of the sequences lengths

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing "sequence".
    maxlen_ref : int
        Value of maxlen, shown as a red line.
    """
    lengths = df["sequence"].str.len()

    plt.figure(figsize=(8, 4))
    plt.hist(lengths, bins=50, edgecolor="black")
    plt.xlabel("Sequence length")
    plt.ylabel("Sequence number")
    plt.title("Distribution of sequence length")
    plt.axvline(maxlen_ref, color="red")

    plt.show()

In [13]:
nuc_chars = ['*', 'A', 'C', 'G', 'U']
nuc_to_idx = {c: i for i, c in enumerate(nuc_chars)}
idx_to_nuc = {i: c for i, c in enumerate(nuc_chars)}
vocab_size = len(nuc_to_idx)

num_outputs = 2

In [ ]:
# LOAD + ENCODING
df_dms, df_2a3, sequences, encoded_seqs = load_data_and_encode(
    "train_data_QUICK_START.csv", nuc_to_idx, 2000
)

# TARGETS + PADDING + SPLIT
y = build_targets(df_dms, df_2a3, 206)
print("y :", y.shape, "- NaN :", np.isnan(y).sum())

X, x_train, x_test, y_train, y_test = prepare_data(
    encoded_seqs,
    y,
    206
)

In [ ]:
# CREATE MODEL
model = create_model(
    maxlen=206,
    embed_dim=64,
    vocab_size=vocab_size,
    num_outputs=2,
    num_heads=4,
    dense_neuron=64,
    num_block_treatment=3,
)
model.summary()


In [ ]:
history = train_model(model, x_train, y_train, batch_size=32, epoch_num=100, patience=10)

In [ ]:
plot_history(history, "model1")

In [18]:
test_results = evaluate_model(model, x_test, y_test, model_name="model1")

13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 400ms/step
[model1] Test MAE: 0.2211 — DMS_MaP: 0.1880 — 2A3_MaP: 0.2541


---------------------------------------------------------------------------------bpp----------------------------------------------------------------------------


In [37]:
import os
if not os.path.exists("bpp_2000"):
    raise FileNotFoundError(
        "Dossier bpp_2000/ manquant. Voir README pour instructions."
    )
import os

In [ ]:
print(len(os.listdir("bpp_2000")))

In [39]:
def create_model_bpp(maxlen, embed_dim, vocab_size, num_outputs, num_heads, dense_neuron, num_block_treatment):
    inputs = keras.Input(shape=(maxlen,), name="seq")
    bpp_input = keras.Input(shape=(maxlen, 1), name="bpp")

    x_emb = keras_hub.layers.TokenAndPositionEmbedding(
        vocabulary_size=vocab_size,
        sequence_length=maxlen,
        embedding_dim=embed_dim,
    )(inputs)

    bpp_proj = layers.Dense(embed_dim)(bpp_input)
    x = x_emb + bpp_proj

    for i in range(num_block_treatment):
        x_norm = layers.LayerNormalization(epsilon=1e-6)(x)
        x_atn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x_norm, x_norm)
        x_atn = layers.Dropout(0.15)(x_atn)
        x = x + x_atn

        x_norm2 = layers.LayerNormalization(epsilon=1e-6)(x)
        x_ff = layers.Dense(embed_dim * 4, activation="gelu")(x_norm2)
        x_ff = layers.Dropout(0.15)(x_ff)
        x_ff = layers.Dense(embed_dim)(x_ff)
        x_ff = layers.Dropout(0.15)(x_ff)
        x = x + x_ff

    dens = layers.Dense(dense_neuron)(x)
    outputs = layers.Dense(num_outputs, activation="sigmoid")(dens)

    return keras.Model(inputs=[inputs, bpp_input], outputs=outputs)

In [63]:
def train_model_bpp(model, x_train, bpp_train, y_train, batch_size, epoch_num, patience):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=3e-4),
        loss=masked_mae,
    )

    callbacks = [
        keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_delta=1e-4,
        cooldown=2,
        min_lr=1e-5,
        verbose=1,
    ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
        )
    ]

    history = model.fit(
        [x_train, bpp_train],
        y_train,
        batch_size=batch_size,
        epochs=epoch_num,
        validation_split=0.2,
        callbacks=callbacks,
    )
    return history

In [64]:
def evaluate_model_bpp(model, x_test, bpp_test, y_test, model_name):
    pred = model.predict([x_test, bpp_test])

    mask_all = ~np.isnan(y_test)
    mae_all = np.abs(y_test[mask_all] - pred[mask_all]).mean()

    mask_dms = ~np.isnan(y_test[:, :, 0])
    mae_dms = np.abs(y_test[:, :, 0][mask_dms] - pred[:, :, 0][mask_dms]).mean()

    mask_2a3 = ~np.isnan(y_test[:, :, 1])
    mae_2a3 = np.abs(y_test[:, :, 1][mask_2a3] - pred[:, :, 1][mask_2a3]).mean()

    results = {"mae": mae_all, "mae_dms": mae_dms, "mae_2a3": mae_2a3}
    print(f"[{model_name}] Test MAE: {mae_all:.4f} — DMS_MaP: {mae_dms:.4f} — 2A3_MaP: {mae_2a3:.4f}")
    return results

In [42]:
def read_bpp(chemin, maxlen):
    paire = np.zeros(maxlen)
    f = open(chemin)
    texte = f.read()
    f.close()

    for ligne in texte.split("\n"):
        if ligne == "":
            continue
        morceaux = ligne.split()
        i = int(morceaux[0]) - 1
        j = int(morceaux[1]) - 1
        p = float(morceaux[2])

        if i < maxlen:
            paire[i] = paire[i] + p
        if j < maxlen:
            paire[j] = paire[j] + p

    return paire

In [43]:
bpp_liste = []
manquants = 0

for sid in df_dms["sequence_id"].tolist():
    chemin = "bpp_2000/" + sid + ".txt"
    if os.path.exists(chemin):
        bpp_liste.append(read_bpp(chemin, 206))
    else:
        bpp_liste.append(np.zeros(206))
        manquants = manquants + 1

bpp = np.array(bpp_liste)
bpp = bpp.reshape(len(bpp), 206, 1)
print(bpp.shape, "manquants :", manquants)

(2000, 206, 1) manquants : 0


In [45]:
df_dms, df_2a3, sequences, encoded_seqs = load_data_and_encode(
    "train_data_QUICK_START.csv", nuc_to_idx, 2000
)

y = build_targets(df_dms, df_2a3, 206)

X, x_train, x_test, y_train, y_test = prepare_data(encoded_seqs, y, 206)

In [ ]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(encoded_seqs))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42)

X = keras.utils.pad_sequences(
    encoded_seqs, maxlen=206, padding="post", truncating="post", value=0,
)

x_train, x_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
bpp_train, bpp_test = bpp[idx_train], bpp[idx_test]

print("x_train:", x_train.shape, "| bpp_train:", bpp_train.shape, "| y_train:", y_train.shape)

In [ ]:
ids = df_dms["sequence_id"].tolist()

print("sequences :", len(sequences))
print("bpp :", len(bpp))
print("ids différents :", len(set(ids)))
print("fichiers dans le dossier :", len(os.listdir("bpp_2000")))
print("manquants :", manquants)

In [ ]:
model_bpp = create_model_bpp(
    maxlen=206,
    embed_dim=128,
    vocab_size=vocab_size,
    num_outputs=num_outputs,
    num_heads=4,
    dense_neuron=64,
    num_block_treatment=4,
)
model_bpp.summary()

In [ ]:
history = train_model_bpp(model_bpp, x_train, bpp_train, y_train, batch_size=32, epoch_num=100, patience=5)

In [60]:
print(len(model_bpp.inputs))

2


In [ ]:
plot_history(history, "model_bpp")

In [ ]:
test_results = evaluate_model_bpp(model_bpp, x_test, bpp_test, y_test, model_name="model_bpp")